In [33]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Carregar os dados do CSV
data = pd.read_csv('sprintia.csv')
data = data.drop_duplicates()  # Remove linhas duplicadas

A função `limpeza_texto` converte as letras para minúsculas, remove caracteres não alfabéticos e pontuação, mantendo apenas letras e espaços e remove espaços extras.


In [34]:
# Função para limpar texto
def limpeza_texto(texto):
    texto = texto.lower() 
    texto = re.sub(r'[^a-z\sçãáéíóúêôãà]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip() 
    return texto
# Aplica a limpeza de texto 
data['possivel_problema'] = data['possivel_problema'].apply(limpeza_texto)
data['solucao'] = data['solucao'].apply(limpeza_texto)

A lista `problemas_agrupados` vai armazenar dicionários que são uma combinação de problema, sintoma e solução.
O `groupby` é usado para agrupar os dados com base na coluna `possivel_problema`. Para cada grupo há combinações de soluções e para cada sintomas tem variações.

In [35]:
# Criar uma nova tabela para armazenar os problemas agrupados
problemas_agrupados = []
# Agrupar os dados
for problema, grupo in data.groupby('possivel_problema'):
    for index, row in grupo.iterrows():
        # Gerar  combinações de soluções para cada sintoma
        combinacoes_solucao = [row['solucao']]  # Começa com a solução original

        if "substituir" in row['solucao']:
            combinacoes_solucao.append(row['solucao'].replace("substituir", "trocar"))
            combinacoes_solucao.append(row['solucao'].replace("substituir", "realizar a troca"))
        
        # Gera múltiplas entradas para cada combinação de sintoma e solução
        for sintoma_variacao in [row['sintoma'], row['sintoma'] + " com barulho", row['sintoma'] + " e vibração"]:
            for solucao in combinacoes_solucao:
                problemas_agrupados.append({
                    'cluster': 0, 
                    'problema': problema,
                    'sintoma': sintoma_variacao,
                    'solucao': solucao
                })
# Converter a lista de dicionários em um DataFrame
problemas_agrupados_df = pd.DataFrame(problemas_agrupados)

Foi realizada a vetorização de sintomas de forma quantitativa para calcular similaridades entre eles.

Para que seja mais precisa a busca por similaridades, há a função `encontrar_sintomas_parecidos`, que recebe um sintoma inserido pelo usuário e calcula sua similaridade em relação aos sintomas existentes no dataset. Ela utiliza o cálculo de similaridade de cosseno para determinar o quão próximo os sintomas são entre si.

Enquanto a `encontrar_problemas_solucoes`, busca problemas e soluções que correspondem aos sintomas fornecidos pelo usuário. Para cada sintoma, ela calcula a similaridade e verifica se está dentro do limite de 50%.

In [36]:
# Salvar a tabela problemas agrupados em um novo CSV
problemas_agrupados_df.to_csv('problemas_agrupados.csv', index=False, encoding='utf-8')

# Vetorização dos sintomas do dataset
vectorizer = TfidfVectorizer().fit(data['sintoma'])
vectors = vectorizer.transform(data['sintoma']).toarray()

# Função para buscar sintomas semelhantes e retornar a porcentagem de correspondência
def encontrar_sintomas_parecidos(sintoma_usuario):
    user_vec = vectorizer.transform([limpeza_texto(sintoma_usuario)]).toarray()
    sim = cosine_similarity(user_vec, vectors)
    return sim.flatten()

In [37]:
# Função para buscar problemas e soluções que correspondam a todos os sintomas
def encontrar_problemas_solucoes(sintomas_usuario):
    problemas_encontrados = {}
    porcentagens = []

    for sintoma in sintomas_usuario:
        similaridades = encontrar_sintomas_parecidos(sintoma)
        porcentagem_media = similaridades.max() * 100  # Convertendo para porcentagem
        porcentagens.append(porcentagem_media)

        if porcentagem_media >= 50:  # Se a porcentagem for maior que 50%
            sintoma_index = similaridades.argmax()  # Sintoma mais parecido
            problema = data['possivel_problema'].iloc[sintoma_index]
            solucao = data['solucao'].iloc[sintoma_index]
            if problema in problemas_encontrados:
                problemas_encontrados[problema].append(solucao)  # Adiciona solução se o problema já existe
            else:
                problemas_encontrados[problema] = [solucao]  # Cria nova entrada para o problema

    if porcentagens:
        porcentagem_media_total = sum(porcentagens) / len(porcentagens)  # Calcula a porcentagem média
        return porcentagem_media_total, problemas_encontrados
    else:
        return 0, {}

A função refina a busca com base nos sintomas fornecidos pelo usuário, verificando a porcentagem média de similaridade: 
- Se >= 90%, retorna o problema mais comum e suas soluções.
- Se entre 70% e 90%, retorna todos os problemas encontrados.
- Se entre 50% e 70%, retorna diagnósticos individuais.
- Se < 50%, finaliza a busca.

In [38]:
# Função para refinar a busca de problemas e soluções
def refinar_busca(sintomas_usuario):
    while True:
        porcentagem_media, problemas = encontrar_problemas_solucoes(sintomas_usuario)

        if porcentagem_media >= 90:  # 90% ou 100%
            problema_mais_comum = max(problemas.items(), key=lambda item: len(item[1]))  # Problema com mais soluções
            return {problema_mais_comum[0]: problema_mais_comum[1]}  # Retorna o problema e suas soluções

        elif 70 <= porcentagem_media < 90:  # 70% a 90%
            return problemas  # Retorna problemas agrupados

        elif 50 <= porcentagem_media < 70:  # 50% a 70%
            return {p: s for p, s in problemas.items()}  # Retorna diagnósticos individuais

        else:
            print("Saindo da busca.")
            return {}

In [39]:
# Coleta de sintomas do usuário
sintomas_usuario = []
while True:
    sintoma = input("Digite um sintoma (ou pressione Enter para sair): ").strip()
    if sintoma:
        sintomas_usuario.append(sintoma)
    else:
        break
# Verificar se o usuário inseriu sintomas
if not sintomas_usuario:
    print("Nenhum sintoma foi inserido.")
else:
    # Busca e exibe o resultado
    problemas = refinar_busca(sintomas_usuario)
    
    if problemas:
        # Listar problemas e soluções em uma única resposta
        print("\nResultados encontrados:\n")
        for problema, solucoes in problemas.items():
            solucoes_formatadas = ', '.join(set(solucoes))  # Remove duplicatas nas soluções
            print(f"Problema: {problema.capitalize()}\nSoluções: {solucoes_formatadas.capitalize()}\n")
    else:
        print("Nenhum problema ou solução encontrada.")

Nenhum sintoma foi inserido.


In [40]:
!pip install joblib


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import pickle

modelo_completo = {
    'data' : data,
    'vectorizer': vectorizer,
    'vectors': vectors,
    'limpeza_texto': limpeza_texto,
    'encontrar_sintomas_parecidos': encontrar_sintomas_parecidos,
    'encontrar_problemas_solucoes': encontrar_problemas_solucoes,
    'refinar_busca': refinar_busca
}


with open('modelo_completo.pickle', 'wb') as f:
    pickle.dump(modelo_completo,f)